[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_03/MFM_ch3_jkp_replication/MFM_ch3_jkp_replication.ipynb)

# MFM_ch3_jkp_replication

Lecture 3 case study (Jensen, Kelly and Pedersen, 2023, Is There a Replication Crisis in Finance?): CAPM alphas of 15 US long-short factors built from value-weighted univariate sorts (July 1963 - July 2026), scaled to 10% annual idiosyncratic volatility; OLS and Benjamini-Yekutieli replication rates, themes of the paper (authors' cluster labels), and hierarchical empirical-Bayes posterior alphas.

*Modelling Financial Markets (MFM), Chapter 3: Factor Models and Asset Pricing. Author: Daniel Traian Pele.*

In [ ]:
import os
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Data

Market prices from the course repository (`data/market`); factor and portfolio returns from the Kenneth French Data Library.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

SECTORS = ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']        # from Dec 1998
SECTORS_ALL = SECTORS + ['XLRE', 'XLC']                                           # XLRE 2015, XLC 2018
SECTOR_NAMES = {'XLB': 'Materials', 'XLE': 'Energy', 'XLF': 'Financials', 'XLI': 'Industrials',
                'XLK': 'Technology', 'XLP': 'Cons. staples', 'XLU': 'Utilities', 'XLV': 'Health care',
                'XLY': 'Cons. discretionary', 'XLRE': 'Real estate', 'XLC': 'Communication'}
FACTOR_ETFS = ['MTUM', 'VLUE', 'QUAL', 'USMV', 'IWM', 'IWD', 'IWF']
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'FP', 'SNG', 'H2O', 'SNN', 'EL', 'DIGI', 'TEL']
BVB_NAMES = {'TLV': 'Banca Transilvania', 'SNP': 'OMV Petrom', 'BRD': 'BRD-GSG', 'TGN': 'Transgaz',
             'FP': 'Fondul Proprietatea', 'SNG': 'Romgaz', 'H2O': 'Hidroelectrica', 'SNN': 'Nuclearelectrica',
             'EL': 'Electrica', 'DIGI': 'Digi', 'TEL': 'Transelectrica'}

_CACHE = {}


def read_market(symbol):
    """Daily prices of one asset from the course data."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(symbol):
    """Price used: adjusted close for ETFs and stocks, close for indices."""
    d = read_market(symbol)
    col = 'close' if symbol.endswith('.INDX') or symbol == 'BET' else 'adjusted_close'
    s = d[col].astype(float)
    return s[s > 0].rename(symbol)


def prices(symbols, start=None, end=None):
    """Prices aligned on the common days (multi-asset analyses align prices before taking returns)."""
    p = pd.concat([price(s) for s in symbols], axis=1).dropna()
    return p.loc[start:end]


def log_returns(p):
    """Log returns of an already aligned table (common days)."""
    return np.log(p).diff().dropna()


# =============================================================================
# KENNETH FRENCH DATA LIBRARY
# =============================================================================
FILES = {
    ('ff3', 'M'): 'F-F_Research_Data_Factors_CSV.zip',
    ('ff3', 'D'): 'F-F_Research_Data_Factors_daily_CSV.zip',
    ('ff5', 'M'): 'F-F_Research_Data_5_Factors_2x3_CSV.zip',
    ('ff5', 'D'): 'F-F_Research_Data_5_Factors_2x3_daily_CSV.zip',
    ('mom', 'M'): 'F-F_Momentum_Factor_CSV.zip',
    ('mom', 'D'): 'F-F_Momentum_Factor_daily_CSV.zip',
    ('p25', 'M'): '25_Portfolios_5x5_CSV.zip',
    ('ind10', 'M'): '10_Industry_Portfolios_CSV.zip',
    ('ind30', 'M'): '30_Industry_Portfolios_CSV.zip',
    ('p100', 'M'): '100_Portfolios_10x10_CSV.zip',
    # univariate sorts (first table: value-weighted), Jensen-Kelly-Pedersen case study
    ('ME', 'M'): 'Portfolios_Formed_on_ME_CSV.zip',
    ('BE-ME', 'M'): 'Portfolios_Formed_on_BE-ME_CSV.zip',
    ('OP', 'M'): 'Portfolios_Formed_on_OP_CSV.zip',
    ('INV', 'M'): 'Portfolios_Formed_on_INV_CSV.zip',
    ('E-P', 'M'): 'Portfolios_Formed_on_E-P_CSV.zip',
    ('CF-P', 'M'): 'Portfolios_Formed_on_CF-P_CSV.zip',
    ('D-P', 'M'): 'Portfolios_Formed_on_D-P_CSV.zip',
    ('AC', 'M'): 'Portfolios_Formed_on_AC_CSV.zip',
    ('NI', 'M'): 'Portfolios_Formed_on_NI_CSV.zip',
    ('BETA', 'M'): 'Portfolios_Formed_on_BETA_CSV.zip',
    ('VAR', 'M'): 'Portfolios_Formed_on_VAR_CSV.zip',
    ('RESVAR', 'M'): 'Portfolios_Formed_on_RESVAR_CSV.zip',
    ('PRIOR_12_2', 'M'): '10_Portfolios_Prior_12_2_CSV.zip',
    ('PRIOR_1_0', 'M'): '10_Portfolios_Prior_1_0_CSV.zip',
    ('PRIOR_60_13', 'M'): '10_Portfolios_Prior_60_13_CSV.zip',
}


def _parse_first_table(text, date_len):
    """First table of a French data set: header ',col1,col2...', then rows YYYYMM(DD),values."""
    lines = text.splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if not parts or len(parts[0]) != date_len or not parts[0].isdigit():
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    fmt = '%Y%m' if date_len == 6 else '%Y%m%d'
    df['date'] = pd.to_datetime(df['date'], format=fmt)
    if date_len == 6:
        df['date'] = df['date'] + pd.offsets.MonthEnd(0)
    df = df.set_index('date')
    return df.replace([-99.99, -999.0], np.nan) / 100.0          # percent -> decimal


def french(name='ff5', freq='M'):
    """Fama-French factors / momentum / portfolios, in decimals (0.01 = 1%)."""
    key = (name, freq)
    if key in _CACHE:
        return _CACHE[key]
    url = FRENCH + FILES[key]
    raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode('latin-1')
    df = _parse_first_table(text, 6 if freq == 'M' else 8)
    df.columns = [c.replace('Mom', 'MOM').strip() for c in df.columns]
    _CACHE[key] = df
    return df


def factors(freq='M'):
    """Mkt-RF, SMB, HML, RMW, CMA, RF, MOM and SMB_FF3 over the common period.

    The five-factor SMB (sorts on B/M, profitability and investment) enters FF5;
    SMB_FF3 is the published three-factor SMB (2x3 sorts on size and B/M),
    used in FF3 and Carhart. HML, Mkt-RF and RF are the same in the two data sets.
    """
    f3 = french('ff3', freq)[['SMB']].rename(columns={'SMB': 'SMB_FF3'})
    return pd.concat([french('ff5', freq), french('mom', freq), f3], axis=1).dropna()


# =============================================================================
# REGRESSIONS
# =============================================================================
def ols_hac(y, X, lags=None, const=True):
    """OLS with Newey-West (Bartlett) standard errors. Returns (coef, se, t, resid, r2)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if X.ndim == 1:
        X = X[:, None]
    if const:
        X = np.column_stack([np.ones(len(y)), X])
    n, k = X.shape
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    u = X * e[:, None]
    S = u.T @ u
    for l in range(1, lags + 1):
        w = 1 - l / (lags + 1)
        G = u[l:].T @ u[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(V))
    r2 = 1 - e.var() / y.var()
    return b, se, b / se, e, r2


def grs_test(excess, market_excess):
    """Gibbons-Ross-Shanken (1989) test of alpha = 0 on N assets, one factor.

    Sigma and the factor variance are maximum-likelihood estimators (division by T);
    under i.i.d. Normal residuals, W ~ F(N, T - N - 1) exactly.
    """
    from scipy import stats
    R = np.asarray(excess, float)
    f = np.asarray(market_excess, float)
    T, N = R.shape
    X = np.column_stack([np.ones(T), f])
    B = np.linalg.lstsq(X, R, rcond=None)[0]
    alpha = B[0]
    E = R - X @ B
    Sigma = E.T @ E / T
    mu, s2 = f.mean(), f.var(ddof=0)
    stat = (T - N - 1) / N * (alpha @ np.linalg.solve(Sigma, alpha)) / (1 + mu ** 2 / s2)
    p = 1 - stats.f.cdf(stat, N, T - N - 1)
    return stat, p, alpha, B[1]

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
PALETTE = [MainBlue, IDAred, Forest, Amber, Orange, Purple, Crimson, '#795548', '#17A2B8', '#6F42C1', '#20C997']

# models: FF3 and Carhart use the published three-factor SMB; FF5 uses the five-factor SMB
FF3 = ['Mkt-RF', 'SMB_FF3', 'HML']
CARHART = FF3 + ['MOM']
FF5 = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
FF6 = FF5 + ['MOM']

SEED = 42
MAX_ABS_RET = 0.5        # data-error threshold for the BVB series (daily log return)
END = '2026-07-31'       # last month of the monthly sample used in the chapter (T = 757 months from July 1963)


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def monthly_excess(symbols, start='1999-01-01'):
    """Monthly simple excess returns over RF (one-month T-bill), on the common months."""
    p = prices(symbols).resample('ME').last()
    r = p.pct_change().dropna()
    F = factors('M')
    idx = r.index.intersection(F.index)
    r = r.loc[idx].loc[start:END]
    F = F.loc[r.index]
    return r.sub(F['RF'], axis=0), F


def daily_excess_one(symbol, start=None):
    """Daily excess return of a single asset, on the factor calendar.

    Only days with a price on the previous trading day are kept
    (otherwise the return would span several days and the factors only one).
    """
    s = price(symbol)
    F = factors('D').loc[:END]                         # same sample end as the monthly data
    s = s.loc[s.index.isin(F.index)]
    cal = pd.Series(np.arange(len(F)), index=F.index)
    pos = cal.loc[s.index].values
    r = s.pct_change()
    consecutive = np.r_[False, np.diff(pos) == 1]
    r = r[consecutive].dropna()
    if start:
        r = r.loc[start:]
    Fx = F.loc[r.index]
    return (r - Fx['RF']).rename(symbol), Fx


def daily_excess(symbols, start=None):
    """Compatibility: dictionary {symbol: (excess, factors)} on each asset's calendar."""
    return {s: daily_excess_one(s, start) for s in symbols}

JKP_SIGNALS = {'Size': ('ME', 'Lo 30', 'Hi 30'), 'Book/market': ('BE-ME', 'Hi 30', 'Lo 30'), 'Profitability': ('OP', 'Hi 30', 'Lo 30'), 'Investment': ('INV', 'Lo 30', 'Hi 30'), 'Earnings/price': ('E-P', 'Hi 30', 'Lo 30'), 'Cash flow/price': ('CF-P', 'Hi 30', 'Lo 30'), 'Dividend/price': ('D-P', 'Hi 30', 'Lo 30'), 'Accruals': ('AC', 'Lo 20', 'Hi 20'), 'Net share issues': ('NI', 'Lo 20', 'Hi 20'), 'Market beta': ('BETA', 'Lo 20', 'Hi 20'), 'Variance': ('VAR', 'Lo 20', 'Hi 20'), 'Residual variance': ('RESVAR', 'Lo 20', 'Hi 20'), 'Momentum 12-2': ('PRIOR_12_2', 'Hi PRIOR', 'Lo PRIOR'), 'Reversal 1-0': ('PRIOR_1_0', 'Lo PRIOR', 'Hi PRIOR'), 'Reversal 60-13': ('PRIOR_60_13', 'Lo PRIOR', 'Hi PRIOR')}
JKP_THEMES = {'Size': 'Size', 'Book/market': 'Value', 'Profitability': 'Profitability', 'Investment': 'Investment', 'Earnings/price': 'Value', 'Cash flow/price': 'Value', 'Dividend/price': 'Value', 'Accruals': 'Accruals', 'Net share issues': 'Value', 'Market beta': 'Low risk', 'Variance': 'Low risk', 'Residual variance': 'Low risk', 'Momentum 12-2': 'Momentum', 'Reversal 1-0': 'Short-term reversal', 'Reversal 60-13': 'Investment'}
JKP_THEME_ORDER = ['Value', 'Investment', 'Low risk', 'Profitability', 'Accruals', 'Momentum', 'Size', 'Short-term reversal']

## Analysis

In [ ]:
def jkp_replication(start='1963-07-31'):
    """Jensen, Kelly & Pedersen (2023) on 15 US factors: CAPM alphas with factors scaled to 10% annual
    idiosyncratic volatility (Sec. II.A), OLS replication rate (t >= 1.96), Benjamini-Yekutieli at 5%,
    the paper's themes (JKP_THEMES, Sec. II.B) and the hierarchical empirical-Bayes model with prior mean 0
    (eqs. 21-23, Prop. 4; one region, so tau_s = 0); also the variant with a single theme."""
    from scipy.optimize import minimize
    X = pd.DataFrame({k: french(f, 'M')[lo] - french(f, 'M')[sh] for k, (f, lo, sh) in JKP_SIGNALS.items()})
    X = X.loc[start:END].dropna()
    m = factors('M').loc[X.index, 'Mkt-RF'].values
    T, N = X.shape
    Z = np.column_stack([np.ones(T), m])
    E0 = X.values - Z @ np.linalg.lstsq(Z, X.values, rcond=None)[0]
    Xs = X * (0.10 / np.sqrt(12)) / E0.std(0, ddof=2)               # monthly residual volatility 10%/sqrt(12)
    B = np.linalg.lstsq(Z, Xs.values, rcond=None)[0]
    E = Xs.values - Z @ B
    a = B[0]
    se = np.sqrt((E ** 2).sum(0) / (T - 2) * np.linalg.inv(Z.T @ Z)[0, 0])
    t = a / se
    p = 2 * stats.norm.sf(np.abs(t))
    # Benjamini-Yekutieli at 5%
    o = np.argsort(p)
    ok = p[o] <= np.arange(1, N + 1) / (N * np.sum(1 / np.arange(1, N + 1))) * 0.05
    by = np.zeros(N, bool)
    by[o[:np.max(np.where(ok)[0]) + 1 if ok.any() else 0]] = True
    S = np.cov(E.T, ddof=2) / T                                      # Sigma / T

    def eb(g):
        Mm = (g[:, None] == np.unique(g)[None, :]).astype(float)

        def nll(q):
            tc, tw = np.exp(q)
            return -stats.multivariate_normal.logpdf(a, np.zeros(N), Mm @ Mm.T * tc ** 2 + np.eye(N) * tw ** 2 + S)
        fits = [minimize(nll, np.log(x0), method='Nelder-Mead', options=dict(xatol=1e-8, fatol=1e-10, maxiter=5000))
                for x0 in ([0.003, 0.002], [0.0005, 0.003], [0.003, 0.0003])]
        best = min(fits, key=lambda z: z.fun)
        tc, tw = np.exp(best.x)
        O = Mm @ Mm.T * tc ** 2 + np.eye(N) * tw ** 2
        P = np.linalg.inv(np.linalg.inv(O) + np.linalg.inv(S))       # Prop. 4, with Sigma / T
        pm = P @ np.linalg.solve(S, a)
        ps = np.sqrt(np.diag(P))
        return dict(tau_c=tc, tau_w=tw, post_mean=pm, post_sd=ps, z=pm / ps, loglik=-best.fun)
    theme = [JKP_THEMES[c] for c in X.columns]
    g = np.array([JKP_THEME_ORDER.index(x) for x in theme])
    r = eb(g)
    r1 = eb(np.zeros(N, int))                                        # variant: a single theme
    k = len(np.unique(g))
    tab = pd.DataFrame({'theme': theme, 'alpha': a, 'se': se, 't': t, 'by': by, 'post_mean': r['post_mean'],
                        'post_sd': r['post_sd'], 'z': r['z']}, index=X.columns)
    summ = dict(T=T, start=str(X.index[0].date()), end=str(X.index[-1].date()), k=k,
                tau_c=r['tau_c'], tau_w=r['tau_w'],
                rate_ols=float(np.mean(t >= 1.96)), rate_by=float(by.mean()),
                by_crit_t=float(np.min(np.abs(t[by]))) if by.any() else None,
                rate_eb=float(np.mean(r['z'] >= 1.96)),
                rate_eb_one_theme=float(np.mean(r1['z'] >= 1.96)), tau_c_one_theme=r1['tau_c'],
                tau_w_one_theme=r1['tau_w'])
    return tab, summ


def fig_jkp_replication():
    tab, s = jkp_replication()
    order = [nm for nm in JKP_THEME_ORDER if nm in set(tab['theme'])]
    d = pd.concat([tab[tab['theme'] == th].sort_values('alpha') for th in order])
    y = np.arange(len(d))[::-1].astype(float)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.3, 3.6), sharey=True, gridspec_kw=dict(width_ratios=[1.25, 1]))
    h = 0.17
    ax1.errorbar(d['alpha'] * 100, y + h, xerr=1.96 * d['se'] * 100, fmt='o', ms=3.5, color=MainBlue, lw=0.9,
                 capsize=0, label='OLS alpha, 95% confidence interval')
    ax1.errorbar(d['post_mean'] * 100, y - h, xerr=1.96 * d['post_sd'] * 100, fmt='s', ms=3.5, color=IDAred, lw=0.9,
                 capsize=0, label='Empirical Bayes: posterior mean, 95% credible interval')
    ax1.axvline(0, color=Gray, lw=0.6)
    ax1.set_xlabel('Alpha, % per month (scaled)')
    ax2.barh(y + h, d['t'], 2 * h, color=MainBlue)
    ax2.barh(y - h, d['z'], 2 * h, color=IDAred)
    ax2.axvline(1.96, color=Forest, ls='--', lw=0.9, label='t = 1.96 (single test, 5%)')
    ax2.axvline(s['by_crit_t'], color=Purple, ls=':', lw=1.2,
                label=f"t = {s['by_crit_t']:.2f} (Benjamini-Yekutieli, 5%)")
    ax2.axvline(0, color=Gray, lw=0.6)
    ax2.set_xlabel('OLS t-statistic / posterior z')
    ax1.set_yticks(y, d.index)
    ax1.tick_params(axis='y', labelsize=8)
    # theme separators and names
    th = d['theme'].values
    for i in range(1, len(d)):
        if th[i] != th[i - 1]:
            for ax in (ax1, ax2):
                ax.axhline(y[i] + 0.5, color=Gray, lw=0.4, ls='-')
    xr = 1.5 * max(d['t'].max(), d['z'].max())
    ax2.set_xlim(0, xr)
    for nm in order:
        yy = y[th == nm]
        ax2.text(xr, yy.mean(), nm.replace(' and ', '\nand '), ha='right', va='center', fontsize=7,
                 color='black', style='italic')
    ax1.set_title(f"CAPM alphas of 15 US factors, {s['start'][:4]}-{s['end'][:4]}", fontsize=9, loc='left')
    ax2.set_title('OLS t versus posterior z', fontsize=9, loc='left')
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2, frameon=False)
    plt.tight_layout()
    save_fig('ch3_jkp_replication')
    return dict(table=tab.reset_index(names='factor').round(5).to_dict(orient='records'), summary=s)

## Run

In [ ]:
res = fig_jkp_replication()
print(res['summary'])
pd.DataFrame(res['table'])

![ch3_jkp_replication](./ch3_jkp_replication.png)

Output: `ch3_jkp_replication.pdf`